# OceanCleanup System 03 — Optimization Example

This notebook adapts the multi-objective optimization workflow from the lecture "System design principles" from 15.09.2026 and the example workbook: `EXAMPLE_Artificial_Reef.ipynb` to the OceanCleanup System 03 offshore plastic-collection barrier. It follows the same structure:

- Define design variables and bounds
- Specify multiple objective functions and preference mappings
- Build constraints
- Run a Genetic Algorithm (GA) using two (potentially three) aggregation paradigms
- Visualize preference functions and optimization results

## Problem

System 03 is a passive, offshore plastic-collection system: a long U-shaped floating barrier towed slowly through a garbage patch by two vessels, funneling floating debris toward a retention zone at the apex. Unlike a fixed coastal structure, its design is a trade-off between capturing as much plastic as possible and avoiding harm to marine life, all while remaining operable by a small crew at reasonable cost.

Design choices — how deep its skirt hangs below the surface, how far apart the two towing vessels hold the ends, how fast the system is towed, how fine its screen mesh is, and how many systems are deployed — all affect capture rate, bycatch risk, fishing-ground blockage, and cost. The barrier length itself is treated as fixed **(maybe this need so be adapted in the future when we want to experiment with the vessel spacing )**.

**Note:** Values in the following implementation are placeholders and not researched values.

## Importing Required Packages


In [ ]:
# Import libraries
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import pchip_interpolate
from scipy.optimize import minimize

# Define default plotting parameters
plt.rcParams['font.size'] = '10'
plt.rcParams['savefig.dpi'] = 300

# Import local module for genetic algorithm
from genetic_algorithm_pfm import GeneticAlgorithm

## Design Variables and Bounds

The System 03 design is parameterized using five design variables (four continuous, one integer). They follow from the stakeholder objectives shown below: each variable influences at least one objective, and most influence several, which is what makes this a trade-off problem. The variables are numbered from left to right as in the scheme.

![Stakeholder objectives and design variables](../Our%20project/Stakeholder_Objectives_and_optimisation/stakeholder_objective_variable_scheme.png)

| Variable | Description | Unit | Type |
|----------|-------------|------|------|
| `x1` | Skirt depth below waterline | m | Continuous |
| `x2` | Distance between support vessels (vessel spacing) | m | Continuous |
| `x3` | Number of systems deployed | – | Integer |
| `x4` | Towing speed | m/s | Continuous |
| `x5` | Mesh / screen size | mm | Continuous |

Bounds and values below are placeholder estimates — replace with values we can justify from project research:

- `x1` skirt depth: a deeper skirt catches more submerged plastic but adds material, drag, and contact with animals in the water column.
- `x2` vessel spacing: a wider mouth captures more plastic but needs more towing force and closes more sea area, meaning fishing area to fishers.
- `x3` number of systems: more systems remove more plastic in total, but raise the total cost, the area closed to fishers, and the ecological contact. It must be a whole number, so it is marked `'int'` in `var_type_mixed`.
- `x4` towing speed: a higher speed captures more per hour but raises drag and fuel cost, gives animals less time to avoid the barrier, and lets small fragments escape.
- `x5` mesh size: a finer mesh retains more microplastic but increases fouling and the risk of trapping small marine life.

`X_irl` holds a rough reference design of today's situation (one System 03, so `x3 = 1`) for later comparison against the optimized solutions — update it once you have better sources.

In [ ]:
# Define the names of variables for later use in plotting and analysis
'''
All of those values are place holders: put in own values.
'''
design_variables = (
    ('x1', 'Skirt depth below waterline',      'm'),
    ('x2', 'Distance between support vessels', 'm'),
    ('x3', 'Number of systems deployed',       '-'),
    ('x4', 'Towing speed',                     'm/s'),
    ('x5', 'Mesh / screen size',               'mm')
)

# Fixed system parameter - NOT a design variable (use it in the objective functions where needed)
barrier_length = 2500   # m, total length of the barrier (about 2.5 km according to The Ocean Cleanup website -> 2024)

# set bounds for all variables
b1 = [1, 6]          # x1 skirt depth [m]
b2 = [200, 2000]     # x2 distance between vessels (must stay below barrier_length) [m]
b3 = [1, 10]         # x3 number of systems [-]
b4 = [0.2, 1.0]      # x4 towing speed [m/s]
b5 = [10, 20]        # x5 mesh size [mm]
bounds = [b1, b2, b3, b4, b5]

# type of each variable: 'real' = continuous, 'int' = whole number (used in the GA options later)
var_type_mixed = ['real', 'real', 'int', 'real', 'real']

X_irl = [4, 1200, 1, 0.5, 10] # real world values --> place holders, put in later
plot_irl = True  # Can be True or False --> plot the IRL point or not

## Constraints

Constraints ensure that all solutions remain physically feasible. For this artificial reef model, one physical constraint is considered: **the reef has a minimum distance from the beach**. This constraint, `constraint_1`, is a simple geometric relation corresponding to the distance `x5` shown in the figure above. A constraint function may use as many or as few design variables as necessary.

Each *constraint function* must be expressed as an algebraic expression of the form:

$$g(\mathbf{x}) \leq 0 \quad \text{(feasible)}$$

> **Important:** The constraint function should return only the value of $g(\mathbf{x})$ — do *not* encode the inequality inside the function. The inequality type (`'ineq'`) is specified separately when registering the constraint in `cons`, and the optimiser handles the comparison.

The constraints are registered as follows:

```python
cons = [['ineq', constraint_1], ['ineq', constraint_2]]

In [ ]:
def constraint_1(variables):
    """Cable Tension:

    :return: 1-D array (length n) with constraint values; arr>0 means violation for 'ineq' constraints.
    """
    x1 = variables[:, 0]
    x2 = variables[:, 1]
    x3 = variables[:, 2]
    x4 = variables[:, 3]
    x5 = variables[:, 4]
    max_tension = 30      # maximum admisible tension of the cable [kN] !!! REVIEW
    Cd = 1 #Hydrodynamic drag coefficient.
    
    return Cd * (barrier_length * x1) * x4 ** 2 * (x2 / barrier_length) * (1 / x5) # If the force exceeds the max tension it can broke.

def constraint_2(variables):
    """Maximum volume:

    :return: 1-D array (length n) with constraint values; arr>0 means violation for 'ineq' constraints.
    """
    x1 = variables[:, 0]
    x2 = variables[:, 1]
    x3 = variables[:, 2]
    x4 = variables[:, 3]
    x5 = variables[:, 4]

    max_volume = 1540 #[m3], maximum theoretical volumetric capacity.
    Kvol = 0.38 #Geometric scaling constant that translates the bounding box of the span into the actual pool volume.
    
    return max_volume - (Kvol * x1 * x2 * ((barrier_length**2) * (x2 ** 2)) ** (1/2)) # < 0  # Max volume: if the result is negative, it means the capacity of that the waste bag can take is exceeded.

def constraint_3(variables):
    """Vessel autonomy constraint

    :return: 1-D array (length n) with constraint values; arr>0 means violation for 'ineq' constraints.
    """
    x1 = variables[:, 0]
    x2 = variables[:, 1]
    x3 = variables[:, 2]
    x4 = variables[:, 3]
    x5 = variables[:, 4]

    K_base = 1740.74 #[L/(system * hour * (m/s)**3)]
    K_weight = 668.80 #[L/(system * hour * (m/s)**2)]

    Fuel_max = 800 #L
    
    return - ((K_base * x3 * x4 ** 3) + (K_weight * 0.50 * (x4 ** 2) * x3)) + Fuel_max # < 0  # If the result is negative, we don't have enough fuel to the system.

cons = [['ineq', constraint_1], ['ineq', constraint_2], ['ineq', constraint_3]]

## Objective Functions

This model optimises five objectives, each representing one stakeholder from the stakeholder analysis (see the scheme above).

| Stakeholder | Objective | Unit | Variables | Goal |
|-------------|-----------|------|-----------|-----------|
| Investors / Donors | Annual cost | M€/yr | x1, x2, x3, x4 | minimise |
| The Ocean Cleanup | Plastic removal | t/yr | x1, x2, x3, x4, x5 | maximise |
| North Pacific commercial fishers | Fishing-area interference | km² | x2, x3 | minimise |
| Marine conservation advocates | Ecological risk (bycatch) | animals/yr | x1, x2, x3, x4, x5 | minimise |
| Citizens | Future plastic fragmentation | t/yr | x2, x3, x4, x5 | minimise |

Each objective is a simple, physically motivated expression of the design variables. The physical constants are collected in **one cell below**, so they can be replaced by researched values later without touching the functions. Constants that are shared between objectives (e.g. plastic density, plastic size distribution) are defined only once, so the objectives stay consistent with each other.

**1. Cost (Investors / Donors).** Annual cost of all systems = number of systems × (fixed vessel charter + annualised barrier material + fuel). Fuel follows from the towing power, which is the drag force times the speed:

$$P_{tow} = \frac{\tfrac{1}{2}\rho_w C_D \, x_1 x_2 \, x_4^3}{\eta_{prop}}$$

The frontal area of the skirt is approximated by skirt depth × mouth width ($x_1 x_2$). Because power scales with $x_4^3$, the cost rises quickly at high towing speeds.

**2. Plastic removal (The Ocean Cleanup).** Plastic that meets the system (surface density × mouth width × speed × operating time), multiplied by three efficiency factors between 0 and 1:
- *depth factor* $1 - e^{-x_1/d_0}$: plastic concentration decreases with depth, so a deeper skirt adds less and less;
- *speed retention* $1 / \left(1 + (x_4/v_{crit})^n\right)$: above a critical speed, plastic is pushed under the skirt and escapes. Together with the linear term in $x_4$, this gives an **optimal towing speed**;
- *mesh retention* $e^{-x_5/\lambda}$: the fraction of the plastic mass that is larger than the mesh opening. It uses an exponential size distribution with characteristic size $\lambda$, so a finer mesh retains more.

**3. Fishing-area interference (Fishers).** The sea area that is closed to fishing around each system. The U is approximated by two straight arms of $L/2$, so its length in the towing direction is $h = \sqrt{(L/2)^2 - (x_2/2)^2}$. A safety buffer $b$ is added on all sides: $A = x_3 (x_2 + 2b)(h + 2b)$.

**4. Ecological risk (Marine conservation advocates).** Expected bycatch = swept water volume ($x_1 x_2 x_4 T$) × density of vulnerable animals × probability an animal cannot escape × mesh factor. The escape probability decreases with speed ($1 - e^{-x_4/v_{avoid}}$ is the probability of *not* escaping), and a finer mesh traps more small organisms ($x_{5,ref}/x_5$).

**5. Future plastic fragmentation (Citizens).** Microplastic that is created by, or passes through, the system: plastic met × fraction smaller than the mesh ($1 - e^{-x_5/\lambda}$, the complement of the mesh retention in objective 2) × a fragmentation factor that increases with towing speed ($k \, x_4^2$, from turbulence and abrasion). This objective deliberately **conflicts** with plastic removal: more systems, a wider mouth and a higher speed catch more plastic but also produce more fragments.

In [ ]:
# ---------------------------------------------------------------------------------------
# Constants for the objective functions
# ALL VALUES ARE PLACEHOLDERS --> replace with researched values
# ---------------------------------------------------------------------------------------

# General operation (used in several objectives)

uptime            = 0.7                        # fraction of the year the system is actively towing [-]
T_op              = uptime * 365 * 24 * 3600   # operational time per year in seconds [s]
rho_water         = 1025                       # sea water density [kg/m3]

# Plastic in the garbage patch (shared by objectives 2 and 5 -> keeps them consistent)

plastic_density   = 5e-5    # floating plastic mass per sea surface area [kg/m2] (= 50 kg/km2)
plastic_size_char = 50      # characteristic size of the plastic size distribution, lambda [mm]

# Objective 1 - Cost

charter_cost      = 10e6    # fixed cost per system: two vessels + crew per year [EUR/yr]
material_cost     = 200     # barrier / skirt material cost per m2 of skirt [EUR/m2]
barrier_lifetime  = 5       # lifetime of the barrier, used to annualise material cost [yr]
drag_coeff        = 1.2     # drag coefficient of the barrier/skirt [-]
prop_efficiency   = 0.6     # propulsion efficiency of the towing vessels [-]
fuel_cost         = 0.25    # fuel cost per kWh of delivered towing energy [EUR/kWh]

# Objective 2 - Plastic removal

plastic_depth     = 1.5     # e-folding depth of plastic concentration in the water column, d0 [m]
v_critical        = 0.8     # towing speed above which plastic escapes under the skirt [m/s]
escape_steepness  = 4       # how sharply retention drops above v_critical, n [-]

# Objective 3 - Fishing-area interference

safety_buffer     = 500     # exclusion buffer around the system in which fishing is not allowed, b [m]

# Objective 4 - Ecological risk

animal_density    = 1e-9    # density of vulnerable animals in the water column [1/m3]
v_avoid           = 0.4     # characteristic speed at which animals can still avoid the barrier [m/s]
mesh_ref          = 10      # reference (finest) mesh size for the mesh factor [mm]

# Objective 5 - Future plastic fragmentation

frag_factor       = 0.1     # fraction of passing plastic that fragments at a towing speed of 1 m/s [-] 

In [ ]:
# Define objective functions

def objective_function_1(x1, x2, x3, x4, x5):
    '''
    Cost function (Investors / Donors).

    :return: float of annual cost of all systems in million EUR per year.
    '''

    material = material_cost * barrier_length * x1 / barrier_lifetime            # [EUR/yr] annualised skirt material
    power    = 0.5 * rho_water * drag_coeff * x1 * x2 * x4**3 / prop_efficiency  # [W] towing power
    fuel     = power * T_op / 3.6e6 * fuel_cost                                  # [EUR/yr] J -> kWh -> EUR

    return x3 * (charter_cost + material + fuel) / 1e6


def objective_function_2(x1, x2, x3, x4, x5):
    '''
    Plastic-removal function (The Ocean Cleanup).

    :return: float of plastic removed by all systems in tonnes per year.
    '''

    encountered     = plastic_density * x2 * x4 * T_op                   # [kg/yr] plastic in front of one system
    depth_factor    = 1 - np.exp(-x1 / plastic_depth)                    # [-] share of plastic above the skirt depth
    speed_retention = 1 / (1 + (x4 / v_critical)**escape_steepness)      # [-] share not escaping under the skirt
    mesh_retention  = np.exp(-x5 / plastic_size_char)                    # [-] share of plastic mass larger than the mesh

    return x3 * encountered * depth_factor * speed_retention * mesh_retention / 1000


def objective_function_3(x1, x2, x3, x4, x5):
    '''
    Fishing-area interference function (North Pacific commercial fishers).

    :return: float of sea area closed to fishing in km2.
    '''

    u_length = np.sqrt((barrier_length / 2)**2 - (x2 / 2)**2)            # [m] length of the U in towing direction

    return x3 * (x2 + 2 * safety_buffer) * (u_length + 2 * safety_buffer) / 1e6


def objective_function_4(x1, x2, x3, x4, x5):
    '''
    Ecological-risk function (Marine conservation advocates).

    :return: float of expected bycatch in animals per year.
    '''

    swept_volume = x1 * x2 * x4 * T_op                                   # [m3/yr] water volume passed by one system
    p_no_escape  = 1 - np.exp(-x4 / v_avoid)                             # [-] probability an animal cannot avoid the barrier
    mesh_factor  = mesh_ref / x5                                         # [-] finer mesh traps more small organisms

    return x3 * animal_density * swept_volume * p_no_escape * mesh_factor


def objective_function_5(x1, x2, x3, x4, x5):
    '''
    Future plastic-fragmentation function (Citizens).

    :return: float of microplastic created/released by all systems in tonnes per year.
    '''
    
    encountered      = plastic_density * x2 * x4 * T_op                  # [kg/yr] plastic in front of one system
    passing_fraction = 1 - np.exp(-x5 / plastic_size_char)               # [-] share of plastic mass smaller than the mesh
    fragmentation    = frag_factor * x4**2                               # [-] share that fragments, grows with speed

    return x3 * encountered * passing_fraction * fragmentation / 1000


# Define the list of objectives with their corresponding names, units and stakeholders for later use in plotting and analysis
objectives = [
    (objective_function_1, "Cost",                 "M€/yr",      "Investors / Donors"),
    (objective_function_2, "Plastic Removal",      "t/yr",       "The Ocean Cleanup"),
    (objective_function_3, "Fishing Interference", "km2",        "Commercial Fishers"),
    (objective_function_4, "Ecological Risk",      "animals/yr", "Conservation Advocates"),
    (objective_function_5, "Fragmentation",        "t/yr",       "Citizens"),
]

As in the reef example, the attainable minimum and maximum of each objective is computed as a sanity check. If a range is physically unreasonable, revisit the constants above. The ranges also define the interval over which the preference curves are drawn. The value of the reference design `X_irl` is printed for comparison.

Note: `minimize` treats `x3` (number of systems) as continuous here. This is fine for finding the range, because the extremes lie at the integer bounds 1 and 10.

In [ ]:
# Finding min and max for each objective using scipy's minimize function, starting from the midpoint of the bounds
objective_minmax = {}  # Dictionary to store the min and max values for each objective
midpoints = [np.mean(b) for b in bounds]

for idx, (obj_func, name, unit, stakeholder) in enumerate(objectives):
    wrapped = lambda x, sign=1: sign * obj_func(*x)

    min_val =  minimize(wrapped, x0=midpoints, bounds=bounds, method='L-BFGS-B').fun
    max_val = -minimize(lambda x: wrapped(x, sign=-1), x0=midpoints, bounds=bounds, method='L-BFGS-B').fun
    irl_val = obj_func(*X_irl)

    objective_minmax[name] = min_val, max_val
    print(f"  Objective {idx+1} {name:<21}:  min = {min_val:>12,.2f}   max = {max_val:>12,.2f}   IRL = {irl_val:>10,.2f}  {unit}")

## Weights

When combining multiple objectives into a single score, each objective is assigned a weight **$w_i$** reflecting its relative importance. All weights must sum to one:

$$\sum_{i=1}^{5} w_i = 1$$

| Weight | Stakeholder | Objective |
|--------|-------------|-----------|
| $w_1$ | Investors / Donors | Annual Cost |
| $w_2$ | The Ocean Cleanup | Plastic Removal |
| $w_3$ | North Pacific commercial fishers | Fishing-area interference |
| $w_4$ | Marine conservation advocates | Ecological risk (bycatch) |
| $w_5$ | Citizens | Future plastic fragmentation |

In this model, all weights are initialised equally ($w_i = \frac{1}{5} = 0.2$) for simplicity.

In [ ]:
# Weights per stakeholder — must sum to 1. Before game all weights were set equal.
#                   cost    PR   fish  eco   F.P.F
weights_before =    [0.20, 0.20, 0.20, 0.20, 0.20]
weights_after =     [0.30, 0.35, 0.15, 0.15, 0.05]

weights = weights_before  # set to which weight to use

# Verify the weights sum to 1 (within numerical tolerance)
assert np.isclose(sum(weights), 1.0), f"Weights must sum to 1, got {sum(weights)}"

## Preference Curves and Preference Functions

Each objective value is mapped to a preference score between 0 and 100 using a *preference curve*, which represents a stakeholder's preference towards their objective. This allows objectives with different units and scales to be compared and combined, where 0 indicates the least desirable outcome and 100 the most desirable.

The curves are defined by a set of control points interpolated using `pchip_interpolate` from the SciPy package, which produces smooth curves that pass exactly through each point. Only the shape of the matters — so no analytical function is needed. Creating preference curves using interpolation is a convenient and flexible approach. Start simple and increase complexity only if necessary; aim to capture the essential behaviour with as few points as possible. The prefence curves are then plotted.

For this reef model, four objectives are rated linearly in the technical cycle. The exception is sediment trapping (objective 3), which follows a non-linear preference: too little trapping fails to widen the beach, but too much blocks sediment from reaching downstream beaches — so an intermediate value is preferred.

Each preference curve is then combined with its corresponding objective function to form a *preference function*, which maps design variables directly to a preference score.

In [ ]:
# Define preferences
prefs_before = [[[150000, 4250000],             [100, 0]],          # preference cost
                [[8, 350],                      [0, 100]],          # preference plastic removal
                [[0, 80000, 150000, 375000],    [0, 100, 50, 0]],   # preference fishing-area interference
                [[2, 9.5],                      [100, 0]],          # preference ecological risk (bycatch)
                [[12500000, 250000000],                [0, 100]]]   # preference future plastic fragmentation

pref_after = [[[150000, 2000000, 4250000],      [100, 80, 0]],
                [[8, 120, 350],                 [0, 60, 100]],
                [[0, 80000, 150000, 375000],    [0, 100, 50, 0]],
                [[2, 9.5],                      [100, 0]],
                [[12500000, 250000000],                [0, 100]]]

prefs = prefs_before  # set to which preferences to use


# Preference curve values for plotting
# Generate objective value ranges 
obj_vals_list = []
for _, name, _, _ in objectives:
    obj_vals = np.linspace(*objective_minmax[name])
    obj_vals_list.append(obj_vals)

# Generate preference values using interpolation
pref_vals_list = []
for pref, obj_vals in zip(prefs, obj_vals_list):
    pref_vals = pchip_interpolate(pref[0], pref[1], obj_vals)
    pref_vals_list.append(pref_vals)

# Plotting the preference curves 
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, (obj_vals, p_vals), (_, name, unit, stakeholder) in zip(axes.flat, zip(obj_vals_list, pref_vals_list), objectives): # Loop to plot each preference curve in its own subplot
    ax.plot(obj_vals, p_vals, color='black')
    ax.set_xlim(min(obj_vals), max(obj_vals))
    ax.set_ylim(0, 100)
    ax.set_title(stakeholder)
    ax.set_xlabel(f'{name} [{unit}]')
    ax.set_ylabel('Preference score')
    ax.grid(linestyle='--')

axes.flat[-1].set_visible(False)  # hide empty 6th subplot
fig.tight_layout()
# plt.savefig('preference_curves.png')  # Uncomment this line to save the figure as a PNG file
plt.show()


# Defines preference functions that convert raw objective values to 0-100 preference scores
def pref_func_1(x1, x2, x3, x4, x5):
    cost = objective_function_1(x1, x2, x3, x4, x5)
    return pchip_interpolate(prefs[0][0], prefs[0][1], cost)

def pref_func_2(x1, x2, x3, x4, x5):
    ride = objective_function_2(x1, x2, x3, x4, x5)
    return pchip_interpolate(prefs[1][0], prefs[1][1],PR)

def pref_func_3(x1, x2, x3, x4, x5):
    sed = objective_function_3(x1, x2, x3, x4, x5)
    return pchip_interpolate(prefs[2][0], prefs[2][1], fish)

def pref_func_4(x1, x2, x3, x4, x5):
    safe = objective_function_4(x1, x2, x3, x4, x5)
    return pchip_interpolate(prefs[3][0], prefs[3][1], eco)

def pref_func_5(x1, x2, x3, x4, x5):
    econ = objective_function_5(x1, x2, x3, x4, x5)
    return pchip_interpolate(prefs[4][0], prefs[4][1], FPF)

pref_funcs = [pref_func_1, pref_func_2, pref_func_3, pref_func_4, pref_func_5]  # list of preference functions for easier handling


def objective(variables):
    """
    Objective function that is fed to the GA. Calles the separate preference functions that are declared above.

    :param variables: array with design variable values per member of the population. Can be split by using array
    slicing
    :return: 1D-array with aggregated preference scores for the members of the population.
    """
    x1 = variables[:, 0]
    x2 = variables[:, 1]
    x3 = variables[:, 2]
    x4 = variables[:, 3]
    x5 = variables[:, 4]

    # calculate the preference scores
    p_1 = pref_func_1(x1, x2, x3, x4, x5)
    p_2 = pref_func_2(x1, x2, x3, x4, x5)
    p_3 = pref_func_3(x1, x2, x3, x4, x5)
    p_4 = pref_func_4(x1, x2, x3, x4, x5)
    p_5 = pref_func_5(x1, x2, x3, x4, x5)
    
    return weights, [p_1, p_2, p_3, p_4, p_5]